# Ranking Grid Cells for Landmine Survey in Antioquia, Colombia

**Author:** Travis Carlson

**Dataset:** *Landmine Detection in Antioquia, Colombia*, the RELand landmine-risk dataset on Hugging Face ([mdulcer10/landmine-detection](https://huggingface.co/datasets/mdulcer10/landmine-detection)), CC BY 4.0, released with Dulce Rubio et al. (2024)

This notebook builds and tests supervised machine learning models that estimate landmine risk for 500 × 500 m grid cells in 15 municipalities of Antioquia, Colombia, all since declared mine-free. Each of the 26,761 cells is labelled 1 if it intersects "at least one confirmed hazardous area or a cleared area where landmines were found during humanitarian demining operations" (dataset card), and 0 otherwise: 430 cells (1.6%) are mine cells. The 63 features are open data on each cell and its municipality: distances to roads, settlements, coca crops and past landmine events, terrain, climate, land use, population and municipal statistics.

**Problem.** Supervised binary classification: predict `mines_outcome` (1 = mines found, 0 = clear) from the 63 features. The models' scores rank a municipality's cells for survey, so the question is how well a model trained on other municipalities ranks the cells of one it has never seen, which is where such a model would be used (Dulce Rubio et al., 2024).

**Approach.** Two classifiers, logistic regression and gradient-boosted trees, both weighted for the 1.6% mine rate. Each municipality is held out in turn: the model is tuned and trained on the other 14 and scores the 15th. The models are compared with two rankings that need no training (a random order, and nearest-past-event-first), trained again without the distance to past events, checked against one random split, and compared with the published RELand benchmark.

**How the sections map to the brief.** Section 2 loads and inspects the data (Task 2); sections 3 and 4 explore and preprocess it (Task 3); section 5 sets out how the models are tested and section 6 trains them (Task 4); sections 7 and 8 evaluate them (Task 5); section 9 shows what the logistic regression weighs; section 10 is the summary (Task 6).

**Data handling.** The two coordinate columns are read once, in section 2.3, for two counts; they are then dropped and never reach a model, a table or a chart. The dataset card warns that predictions "should **not** be used as the sole basis for demining operations without expert human review and validation in the field".

*References cited in this notebook:*
- Dulce Rubio, M., Zeng, S., Wang, Q., Alvarado, D., Moreno Rivera, F., Heidari, H., & Fang, F. (2024). RELand: Risk estimation of landmines via interpretable invariant risk minimization. *ACM Journal on Computing and Sustainable Societies, 2*(2), 1–29. https://doi.org/10.1145/3648437
- Ke, G., Meng, Q., Finley, T., Wang, T., Chen, W., Ma, W., Ye, Q., & Liu, T.-Y. (2017). LightGBM: A highly efficient gradient boosting decision tree. In *Advances in Neural Information Processing Systems 30* (pp. 3146–3154).
- Roberts, D. R., Bahn, V., Ciuti, S., Boyce, M. S., Elith, J., Guillera-Arroita, G., Hauenstein, S., Lahoz-Monfort, J. J., Schröder, B., Thuiller, W., Warton, D. I., Wintle, B. A., Hartig, F., & Dormann, C. F. (2017). Cross-validation strategies for data with temporal, spatial, hierarchical, or phylogenetic structure. *Ecography, 40*(8), 913–929. https://doi.org/10.1111/ecog.02881
- Saito, T., & Rehmsmeier, M. (2015). The precision-recall plot is more informative than the ROC plot when evaluating binary classifiers on imbalanced datasets. *PLOS ONE, 10*(3), Article e0118432. https://doi.org/10.1371/journal.pone.0118432

## 1. Problem and data

### 1.1 Setup

Libraries, the data file and its provenance, fixed settings, and the chart style of the earlier projects. The workflow uses Python, NumPy, Pandas, SciPy, scikit-learn and Matplotlib.

In [1]:
import hashlib
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter
import scipy
from scipy import stats
from scipy.spatial import cKDTree
import sklearn
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import auc, average_precision_score, precision_recall_curve, roc_auc_score
from sklearn.model_selection import GridSearchCV, GroupKFold, StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, StandardScaler

pd.set_option("display.max_columns", 70)
pd.set_option("display.width", 120)

# Files (the data file sits in the same folder as this notebook)
DATA_FILE = Path("landmine_detection.parquet")
FIG_DIR = Path("figures")
FIG_DIR.mkdir(exist_ok=True)

# Provenance of the data file
DATASET_PAGE = "https://huggingface.co/datasets/mdulcer10/landmine-detection"
SOURCE_PATH = "data/train-00000-of-00001.parquet"
REVISION = "30710e43dc447ee95bce30e9ed264f68e454e00c"
DOWNLOADED_ON = "2026-10-18"
EXPECTED_SHA256 = "e18e6a278cc62336ff13a68342f5d8a848aebb5e4b35e0cc64dc654160090337"

# Columns with a special role
COORDS = ["LATITUD_Y", "LONGITUD_X"]  # read once in section 2.3, then dropped
GROUP, TARGET = "Municipio", "mines_outcome"

# Fixed settings
SEED = 0          # one seed for every random step, so each run gives the same results
TOP_SHARE = 0.10  # share of each municipality's cells flagged for survey in the top-10% scores

# Municipality names as the dataset card spells them (the file stores them in capitals)
CARD_NAMES = ["Abejorral", "Alejandría", "Chigorodó", "Cocorná", "El Carmen de Viboral", "Granada",
              "La Unión", "Nariño", "Sabanalarga", "San Carlos", "San Francisco", "San Luis",
              "San Rafael", "San Roque", "Sonsón"]
DISPLAY_NAME = {name.upper(): name for name in CARD_NAMES}

# Chart style: light surface, quiet grid, neutrals for the data and one fixed colour per ranking
SURFACE, INK, INK_2, MUTED = "#fcfcfb", "#0b0b0b", "#52514e", "#898781"
GRID, BASELINE = "#e1e0d9", "#c3c2b7"
COLOURS = {"Random ranking": MUTED, "Distance rule": "#1baf7a",
           "Logistic regression": "#2a78d6", "Gradient-boosted trees": "#eb6834"}

plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "axes.edgecolor": BASELINE, "axes.labelcolor": INK_2, "axes.titlecolor": INK,
    "axes.titlesize": 12, "axes.titleweight": "bold", "axes.titlelocation": "left",
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": GRID, "grid.linewidth": 1, "grid.linestyle": "-",
    "axes.axisbelow": True, "xtick.color": MUTED, "ytick.color": MUTED,
    "text.color": INK, "font.size": 10, "lines.linewidth": 2, "legend.frameon": False,
})


def save_fig(fig, name):
    """Save a figure to the figures folder at print resolution and return its path."""
    path = FIG_DIR / name
    fig.savefig(path, dpi=200, bbox_inches="tight")
    return path


for name, module in [("pandas", pd), ("numpy", np), ("scipy", scipy),
                     ("scikit-learn", sklearn), ("matplotlib", matplotlib)]:
    print(f"{name:<13} {module.__version__}")

pandas        3.0.6
numpy         2.5.4
scipy         1.18.1
scikit-learn  1.9.1
matplotlib    3.11.2


### 1.2 The dataset

The dataset was published on Hugging Face with the RELand study of landmine risk in Colombia (Dulce Rubio et al., 2024), under a CC BY 4.0 licence. Its card describes it this way:
- **Cells:** 15 municipalities of Antioquia, which "have been declared mine-free after comprehensive humanitarian demining operations", divided into 26,761 grid cells of 500 × 500 m.
- **Labels:** a cell is positive if it "intersects at least one confirmed hazardous area or a cleared area where landmines were found during humanitarian demining operations, using data from Colombia's National Mine Action Center (AICMA)"; the other cells "have been thoroughly inspected and declared free of mines". The labels therefore come from completed demining rather than estimates.
- **Features:** 63 "derived from multiple open geospatial, socio-economic, and conflict-related data sources".
- **Testing:** the card suggests "Geographic block cross-validation ... using municipalities as spatial blocks: train on a subset of municipalities and validate on held-out ones, simulating deployment to a new territory."

It meets the brief's dataset rules: tabular, public and licensed for academic use, real rather than synthetic, with a clear binary target, and different from the datasets of Projects 1 and 2 (Colombia's landmine victims register and its record of demining results). It comes from Hugging Face rather than one of the four sources the brief lists; the card links it to its peer-reviewed study and gives its licence, origin and testing protocol.

## 2. Load and inspect

### 2.1 Load the data file

The data file is the dataset's only Parquet file, kept unchanged next to this notebook. Parquet is a tabular format that stores each column with its type, so the file loads without type guessing. Its SHA-256 checksum is compared with the one recorded when it was downloaded, so anyone rerunning the notebook can confirm they are using the same data. The first rows are shown without the two coordinate columns (see Data handling, above).

In [2]:
def file_sha256(path):
    """Return the SHA-256 checksum of a file, reading it in 1 MB chunks.

    Used to confirm that the data file is the snapshot recorded when it was downloaded.
    A different checksum means a newer download or a changed file.
    """
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for chunk in iter(lambda: handle.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()


checksum = file_sha256(DATA_FILE)
print(f"Dataset page:  {DATASET_PAGE}")
print(f"Source file:   {SOURCE_PATH} at revision {REVISION}")
print(f"Downloaded on: {DOWNLOADED_ON}")
print(f"SHA-256:       {checksum}")
if checksum == EXPECTED_SHA256:
    print("Checksum matches the recorded snapshot.")
else:
    print("WARNING: checksum differs from the recorded snapshot; results may not match the report.")

df_raw = pd.read_parquet(DATA_FILE)
print(f"\nLoaded {df_raw.shape[0]:,} rows x {df_raw.shape[1]} columns")
df_raw.drop(columns=COORDS).head()

Dataset page:  https://huggingface.co/datasets/mdulcer10/landmine-detection
Source file:   data/train-00000-of-00001.parquet at revision 30710e43dc447ee95bce30e9ed264f68e454e00c
Downloaded on: 2026-10-18
SHA-256:       e18e6a278cc62336ff13a68342f5d8a848aebb5e4b35e0cc64dc654160090337
Checksum matches the recorded snapshot.

Loaded 26,761 rows x 67 columns


,Municipio,mines_outcome,elevation,rainfall,temperature,population_2012,No. Víctimas por Declaración,airports_dist,seaport_dist,settlement_dist,finance_dist,edu_dist,buildings_dist,waterways_dist,coca_dist,dist_roads_t1,dist_roads_t2,dist_roads_t3,dist_old_mine,soil_texture15_trans1,soil_texture15_trans2,nighttime_lights_2012,hist_mines,retro_pobl_tot,indrural,areaoficialkm2,altura,discapital,pib_percapita_cons,land_use_Agroforestal,land_use_Agrícola,land_use_Conservación de Suelos,land_use_Cuerpo de agua,land_use_Forestal,land_use_Ganadera,land_use_Zonas urbanas,weather_Cuerpo de agua,weather_Cálido húmedo,weather_Cálido húmedo a muy húmedo,weather_Cálido muy húmedo,weather_Cálido seco,weather_Cálido seco a húmedo,weather_Frío húmedo,weather_Frío húmedo a muy húmedo,weather_Frío húmedo y frío muy húmedo,weather_Frío muy húmedo,weather_Muy frío y muy húmedo,weather_Templado húmedo a muy húmedo,weather_Zona urbana,relief_Abanicos aluviales subactuales y recientes,"relief_Abanicos aluvio-torrenciales, abanicoterrazas y glacís subrecientes y antiguos",relief_Cuerpo de agua,relief_Espinazos,relief_Espinazos y colinas,relief_Filas y vigas,relief_Glacís coluvial y coluvios de remoción,relief_Glacís y coluvios de remoción,relief_Lomas y colinas,relief_Plano de inundación,relief_Plano de inundación y terrazas bajas,relief_Terrazas,relief_Terrazas y abanicos terrazas,relief_Vallecitos,relief_Vallecitos coluvio-aluviales,relief_Zona urbana
0,CHIGORODÓ,0,8.321069,76.104863,27.0,24.528168,48175,19.194549,39.655619,5.571759,14.920146,9.712803,0.436186,0.067567,24.901704,17.391914,8.031022,13.885556,14.503779,-0.589368,-0.272355,0.0,12.0,48017.0,0.142951,615,31.0,189.134933,7.396802,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0
1,CHIGORODÓ,0,8.812821,79.864412,27.0,42.277782,48175,17.264668,42.126817,7.420439,15.066498,9.256411,1.380861,0.260521,25.610123,16.740306,10.410064,14.066805,14.769123,-0.588898,-0.223477,0.0,12.0,48017.0,0.142951,615,31.0,189.134933,7.396802,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0
2,CHIGORODÓ,0,7.932982,77.527988,27.0,42.277782,48175,16.895245,42.621386,7.833282,15.144992,9.243559,1.837258,0.152477,25.722864,16.651828,10.880046,13.741084,14.871926,-1.130906,-0.395634,0.0,12.0,48017.0,0.142951,615,31.0,189.134933,7.396802,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0
3,CHIGORODÓ,0,7.177505,80.659765,27.0,46.937914,48175,14.527711,40.988684,10.926273,16.121791,9.885362,0.239270,0.001683,25.190396,16.448207,14.214778,11.350489,16.013928,-1.436796,-0.143082,0.0,12.0,48017.0,0.142951,615,31.0,189.134933,7.396802,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0
4,CHIGORODÓ,0,6.599792,68.592699,27.0,45.104572,48175,13.661605,39.542358,12.317430,16.747234,10.509545,0.386426,0.062822,24.601717,16.586849,15.661970,10.516407,16.704217,-1.513291,-0.173490,0.0,12.0,48017.0,0.142951,615,31.0,189.134933,7.396802,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0


### 2.2 Columns and data types

The 67 columns fall into seven groups. The one-hot columns are the dataset's encoding of three categorical variables (land use, climate zone and relief type): each column is 1 if the cell belongs to that class and 0 otherwise.

In [3]:
ONEHOT_PREFIXES = {"land use": "land_use_", "climate": "weather_", "relief": "relief_"}


def column_group(column):
    """Return the group a column belongs to: coordinates, municipality, target, continuous or a one-hot group."""
    if column in COORDS:
        return "coordinates"
    if column == GROUP:
        return "municipality"
    if column == TARGET:
        return "target"
    for group, prefix in ONEHOT_PREFIXES.items():
        if column.startswith(prefix):
            return f"one-hot: {group}"
    return "continuous feature"


groups_table = (pd.DataFrame({"group": df_raw.columns.map(column_group), "dtype": df_raw.dtypes.astype(str)})
                .groupby("group", sort=False)
                .agg(columns=("dtype", "size"), dtypes=("dtype", lambda s: ", ".join(sorted(set(s))))))
print(f"Columns: {len(df_raw.columns)}; features: "
      f"{groups_table.drop(['coordinates', 'municipality', 'target'])['columns'].sum()}")
groups_table

Columns: 67; features: 63


,columns,dtypes
group,,
coordinates,2,float64
municipality,1,str
target,1,int64
continuous feature,27,"float64, int64"
one-hot: land use,7,int64
one-hot: climate,13,int64
one-hot: relief,16,int64


In [4]:
CONTINUOUS = [c for c in df_raw.columns if column_group(c) == "continuous feature"]
ONEHOT_GROUPS = {group: [c for c in df_raw.columns if c.startswith(prefix)]
                 for group, prefix in ONEHOT_PREFIXES.items()}
ONEHOT = [c for columns in ONEHOT_GROUPS.values() for c in columns]
print(f"Continuous features ({len(CONTINUOUS)}):")
print(df_raw[CONTINUOUS].dtypes.to_string())
for group, columns in ONEHOT_GROUPS.items():
    print(f"\nOne-hot {group} ({len(columns)}): {', '.join(c.removeprefix(ONEHOT_PREFIXES[group]) for c in columns)}")
print(f"\nMunicipalities in the file: {df_raw[GROUP].nunique()}")

Continuous features (27):
elevation                       float64
rainfall                        float64
temperature                     float64
population_2012                 float64
No. Víctimas por Declaración      int64
airports_dist                   float64
seaport_dist                    float64
settlement_dist                 float64
finance_dist                    float64
edu_dist                        float64
buildings_dist                  float64
waterways_dist                  float64
coca_dist                       float64
dist_roads_t1                   float64
dist_roads_t2                   float64
dist_roads_t3                   float64
dist_old_mine                   float64
soil_texture15_trans1           float64
soil_texture15_trans2           float64
nighttime_lights_2012           float64
hist_mines                      float64
retro_pobl_tot                  float64
indrural                        float64
areaoficialkm2                    int64
altura        

The continuous features, as the dataset card describes them (units and sources from the card). Eight of them, marked *(municipality)*, have one value per municipality.

| Column | Description | Unit | Source |
|---|---|---|---|
| `elevation` | Elevation | m above sea level | WorldClim (2023) |
| `rainfall` | Average annual rainfall | mm | WorldClim (2023) |
| `temperature` | Average temperature | °C | WorldClim (2023) |
| `population_2012` | Population estimate for the grid cell (2012) | people | WorldPop (2023) |
| `No. Víctimas por Declaración` | Declared conflict victims *(municipality)* | people | Victims Unit (2019) |
| `airports_dist`, `seaport_dist`, `settlement_dist` | Distance to the nearest airport, seaport, settlement | km | OpenStreetMap (2022) |
| `finance_dist`, `edu_dist`, `buildings_dist` | Distance to the nearest financial institution, school, building | km | OpenStreetMap (2022) |
| `waterways_dist` | Distance to the nearest waterway | km | OpenStreetMap (2022) |
| `coca_dist` | Distance to the nearest coca cultivation area | km | UNODC (2021) |
| `dist_roads_t1`, `dist_roads_t2`, `dist_roads_t3` | Distance to the nearest primary, secondary, tertiary road | km | IGAC (2023) |
| `dist_old_mine` | Distance to the nearest historical landmine accident | km | DAICMA (2022) |
| `soil_texture15_trans1`, `soil_texture15_trans2` | Soil texture components (ILR-transformed) | — | Varón-Ramírez et al. (2022) |
| `nighttime_lights_2012` | Night-time light intensity (2012) | index 0–63 | Li et al. (2020) |
| `hist_mines` | Historical landmine accidents *(municipality)* | incidents | DAICMA (2022) |
| `retro_pobl_tot` | Total population *(municipality)* | people | Acevedo & Bornacelly (2014) |
| `indrural` | Rural index *(municipality)* | 0 = urban, 1 = fully rural | Acevedo & Bornacelly (2014) |
| `areaoficialkm2` | Official area *(municipality)* | km² | Acevedo & Bornacelly (2014) |
| `altura` | Average altitude *(municipality)* | m | Acevedo & Bornacelly (2014) |
| `discapital` | Distance to the departmental capital, Medellín *(municipality)* | km | Acevedo & Bornacelly (2014) |
| `pib_percapita_cons` | GDP per capita at 2009 constant prices *(municipality)* | millions of pesos | Acevedo & Bornacelly (2014) |

**Data quality checks:** empty cells, repeated rows, one class per cell in each one-hot group, values that cannot be negative, and the target.

In [5]:
print(f"Empty cells: {df_raw.isna().sum().sum()}")
print(f"Rows that repeat another row exactly: {df_raw.duplicated().sum()}")

for group, columns in ONEHOT_GROUPS.items():
    classes_per_cell = df_raw[columns].sum(axis=1).value_counts().sort_index()
    print(f"Cells by number of {group} classes: {classes_per_cell.to_dict()}")

unclassed = df_raw[df_raw[ONEHOT_GROUPS["climate"] + ONEHOT_GROUPS["relief"]].sum(axis=1) == 0]
print(f"\nCells with no climate and no relief class: {len(unclassed)}; mine cells among them: "
      f"{unclassed[TARGET].sum()}; by municipality: "
      f"{unclassed[GROUP].map(DISPLAY_NAME).value_counts().to_dict()}")

distance_like = [c for c in CONTINUOUS if "dist" in c or c == "discapital"] + ["population_2012", "nighttime_lights_2012"]
print(f"\nSmallest value among the {len(distance_like)} distance, population and light features: "
      f"{df_raw[distance_like].min().min()}")
assert set(df_raw[GROUP].unique()) == set(DISPLAY_NAME), "unexpected municipality name"

counts = df_raw[TARGET].value_counts().sort_index()
print(f"\nTarget mines_outcome: {counts[0]:,} clear cells (0) and {counts[1]:,} mine cells (1); "
      f"mine rate {counts[1] / len(df_raw):.2%}")

Empty cells: 0


Rows that repeat another row exactly: 0
Cells by number of land use classes: {1: 26761}


Cells by number of climate classes: {0: 27, 1: 26734}
Cells by number of relief classes: {0: 27, 1: 26734}

Cells with no climate and no relief class: 27; mine cells among them: 0; by municipality: {'Sonsón': 20, 'Nariño': 4, 'Abejorral': 3}

Smallest value among the 15 distance, population and light features: 0.0

Target mines_outcome: 26,331 clear cells (0) and 430 mine cells (1); mine rate 1.61%


**What the checks show.**
- No cell is empty and no row repeats another. The data types fit the columns: every feature and the target are numbers (integers for the target, the 0/1 columns, declared victims and official area; decimals for the rest), and the municipality is text.
- Every cell has exactly one land-use class. 27 cells (Sonsón 20, Nariño 4, Abejorral 3) have no climate and no relief class; none is a mine cell. They stay in, with those two groups at 0.
- No distance, population or light value is negative, which the log transform in section 4.2 relies on.
- The target is heavily imbalanced: 430 mine cells against 26,331 clear cells (1.61%).
- Three errors in the dataset card, none of which affects the file: the card heads its tables "30" continuous and "33" one-hot columns and lists "13" relief categories before naming 16, while the file has 27, 36 and 16 (63 features, as the card says); and its loading example names `cmpatino/landmine-detection`, which Hugging Face did not find on 2026-10-18. This notebook reads the downloaded file directly.
- The file stores municipality names in capitals; tables and charts use the card's spelling.

### 2.3 Coordinate checks, then drop the coordinates

This is the one use of the coordinates. Each cell's centre is given as latitude and longitude, and two counts need them: cells that share a centre (repeated cells), and mine cells with another mine cell among their eight neighbours, which shows how clustered mine cells are and matters for how the models are tested (section 3.3). The centres are converted to metres with an equirectangular approximation, close enough at this scale, and a cell's neighbours are those whose centres lie within 750 m: the four side neighbours are about 500 m away, the four corner neighbours about 707 m, and the next ring starts at about 1,000 m. Only the two counts are printed. The coordinates are then dropped.

In [6]:
EARTH_RADIUS_M = 6_371_000
NEIGHBOUR_RADIUS_M = 750


def coordinate_checks(raw):
    """Return the number of repeated cell centres and of mine cells with a mine cell among their eight neighbours."""
    repeated = int(raw.duplicated(COORDS).sum())
    lat = np.deg2rad(raw["LATITUD_Y"].to_numpy())
    lon = np.deg2rad(raw["LONGITUD_X"].to_numpy())
    xy = np.column_stack([lon * EARTH_RADIUS_M * np.cos(lat.mean()), lat * EARTH_RADIUS_M])
    mine_xy = xy[raw[TARGET].to_numpy() == 1]
    # Count each mine cell's mine-cell neighbours; the cell itself is within the radius, so subtract 1
    mine_neighbours = cKDTree(mine_xy).query_ball_point(mine_xy, r=NEIGHBOUR_RADIUS_M, return_length=True) - 1
    return repeated, int((mine_neighbours > 0).sum()), len(mine_xy)


repeated_cells, clustered_mines, n_mines = coordinate_checks(df_raw)
print(f"Cells that share a centre with another cell: {repeated_cells}")
print(f"Mine cells with at least one mine cell among their eight neighbours: "
      f"{clustered_mines} of {n_mines} ({clustered_mines / n_mines:.1%})")

df = df_raw.drop(columns=COORDS)
del df_raw
print(f"\nCoordinates dropped; working table: {df.shape[0]:,} rows x {df.shape[1]} columns")

Cells that share a centre with another cell: 0
Mine cells with at least one mine cell among their eight neighbours: 372 of 430 (86.5%)

Coordinates dropped; working table: 26,761 rows x 65 columns


No two cells share a centre. 372 of the 430 mine cells (86.5%) touch another mine cell: mine cells come in clusters, which section 3.3 returns to.